# Pricing reinsurance with prospicio, beside aggregate

This notebook prices six reinsurance contracts with `prospicio` and sets each result beside the number that Stephen Mildenhall's Python package [`aggregate`](https://github.com/mynl/aggregate) (1.0.1) gives for the same contract. It is written for a reinsurance pricing actuary who already knows `aggregate`. Each section says what the contract is (with the DecL line that builds it in `aggregate`), shows the `prospicio` code, prints the result, and compares it with `aggregate`'s.

| Section | Contract | Source in `aggregate`'s examples |
|---|---|---|
| 1 | A gross book and a five-layer occurrence tower | Reinsurance Functions (`ReTester`) |
| 2 | Ceded and net of a two-layer program, and an 80% share | `net of` and `po` clauses |
| 3 | An annual aggregate limit, then paid reinstatements | Bear and Nemlick Treaty 2 |
| 4 | Aggregate stop loss, and an aggregate deductible | `ReTester` stop loss, Treaty 1 |
| 5 | Retro and swing premium, profit commission, sliding scale | Treaties 4, 5 and 6 |
| 6 | Risk-adjusted premium (loss plus cost of capital) with distortions | `calibrate_distortions` |

To run it, clone the repository and install the released package with `pip install prospicio matplotlib jupyter` (or `uv pip install`; PyPI has prebuilt wheels, so no Rust toolchain is needed), then open this notebook from anywhere in the repository. To run it against unreleased changes, build from source instead (`cd python && maturin develop`, which needs Rust). It does not need `aggregate`: the reference numbers are in `validation/reference/re_pricing_aggregate.csv`, written by `validation/scripts/aggregate_re_pricing.py` (which does need `aggregate` 1.0.1). Nothing here is simulated, so there is no simulation count to set.

**Same grid.** Both packages round the severity to buckets of size `bs`, put the mass beyond a policy limit on the limit itself, and compound with an FFT on `2**16` points, with no normalising (`aggregate`'s `normalize=False`). Wherever a comparison depends on the grid, `prospicio` is given the bucket size that `aggregate` used. Contract terms then act on the same discretized severity in both packages: exactly where a layer boundary is a grid point, and split linearly between the two neighbouring buckets where it is not. So means agree far inside the tolerances below. The bucket is passed by hand only for this parity; section 1 says how to choose it in your own work.

**Tolerances.** They were set after a first run, from the differences that the last section explains. The summary at the end prints the worst difference observed in each class beside its tolerance.

| Quantity | Tolerance | Reason |
|---|---|---|
| Means, probabilities, premiums, commissions, prices | 1e-6 relative | Same grid |
| Standard deviation and CV | 1e-6 relative | Same grid |
| Standard deviation and CV of a share placed off the grid (sections 2 and 4) | 1e-5 relative | `aggregate` re-buckets the ceded amounts |
| Skewness, against the skewness of `aggregate`'s density | 2e-5 relative | FFT rounding noise at the top of the grid enters the third moment (last section) |
| Quantiles | The same grid point | A quantile is a point of the shared grid, and at none of those compared is the cdf within 6.7e-8 of a tie |
| Quantiles of a share placed off the grid | Within one bucket | `aggregate` re-buckets the ceded amounts |
| A stop loss far in the tail (the loss above 2,800 in section 3) | 1e-5 relative | `prospicio` lumps the cut tail on the last point |

Some rows hold by construction rather than by agreement: a calibrated distortion prices `min(X, a)` at the premium it was calibrated to, the CCoC return on capital is the cost of capital the premium was built from, and a maximum reinstatement premium is the deposit times the rates. They are kept as consistency checks, marked `*` in the summary and counted apart from the independent comparisons.

In [ ]:
import csv
import math
from pathlib import Path
from statistics import NormalDist

from prospicio.aggregate import fft, recommend_grid
from prospicio.distributions import Burr, Grid, Lognormal, Mixture, Poisson, claim_count
from prospicio.reinsurance import Layer, LossSensitivePremium, Tower
from prospicio.risk import Distortion, calibrate

POINTS = 2**16  # aggregate's log2 = 16 in every example

# The repository's reference folder, from wherever the notebook runs inside the repository.
REFERENCE = next((p / "validation" / "reference" for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "validation" / "reference").is_dir()), None)
if REFERENCE is None:
    raise FileNotFoundError(
        f"validation/reference not found in {Path.cwd()} or above it: this notebook reads aggregate's numbers "
        "from the repository, so open it from inside a clone of the repository")
with open(REFERENCE / "re_pricing_aggregate.csv") as f:
    AGG = {(r["example"], r["quantity"]): float(r["value"]) for r in csv.DictReader(f)}
print(f"{len(AGG)} reference values from aggregate 1.0.1 in {REFERENCE.parent.name}/{REFERENCE.name}/re_pricing_aggregate.csv")


def money(x):
    return f"{x:,.2f}"


def moments(grid):
    # Mean, standard deviation, CV and skewness of a Grid. Grid has no skewness method,
    # so the third central moment is the one statistic summed here by hand.
    h, p = grid.step, grid.probs
    mean, sd = grid.mean(), math.sqrt(grid.variance())
    m3 = sum((i * h - mean) ** 3 * q for i, q in enumerate(p))
    return mean, sd, sd / mean, m3 / sd**3


def prob_at_least(grid, x):
    # P(X >= x) on a grid whose points may not include x: the cdf half a step below x.
    return 1.0 - grid.cdf(x - grid.step / 2)


def discretize(severity, bs, limit):
    # Round the severity to buckets of size bs, with the policy limit as the last point.
    # Grid.rounding puts all mass beyond its last point there: aggregate's `limit xs 0`.
    grid, _ = Grid.rounding(severity, bs, round(limit / bs) + 1)
    return grid


def mixed_poisson(n, mixing_cv):
    # Gamma-mixed Poisson (aggregate's `mixed gamma cv`): a negative binomial with Var/E = 1 + cv^2 n.
    return claim_count(n, 1 + mixing_cv**2 * n)


def exposure_book(subject, rate, shapes, scale, limit):
    # aggregate's `exposure at rate`, severity `scale * pareto shape - scale` (a Lomax) limited to `limit`.
    # The expected loss of class i is subject * rate; its claim count is solved from it.
    # All classes share one mixing variable, so they are one mixed Poisson over a severity mixture.
    lomax = [Burr(a, 1.0, scale) for a in shapes]
    counts = [s * r / v.lev(limit) for s, r, v in zip(subject, rate, lomax)]
    n = sum(counts)
    return counts, n, Mixture([(c / n, v) for c, v in zip(counts, lomax)])


# Tolerance by class, as (tolerance, unit); see the table above.
TOLERANCES = {
    "mean": (1e-6, "rel"),  # means, probabilities, premiums, commissions, prices
    "sd": (1e-6, "rel"),  # standard deviation and CV
    "share sd": (1e-5, "rel"),  # the same for a share placed off the grid
    "skew": (2e-5, "rel"),  # against the skewness of aggregate's density
    "quantile": (0, "buckets"),  # the same grid point
    "share quantile": (1, "buckets"),  # a share placed off the grid
    "tail": (1e-5, "rel"),  # a stop loss far in the tail
}

COMPARISONS = []  # every comparison, for the summary at the end
SHOWN = []  # those to print at the next report()


def _record(section, name, agg, pro, diff, cls, show, check):
    tol, unit = TOLERANCES[cls]
    row = dict(section=section, name=name, aggregate=agg, prospicio=pro, diff=diff, cls=cls,
               unit=unit, tol=tol, ok=bool(diff <= tol), check=check)
    COMPARISONS.append(row)
    if show:
        SHOWN.append(row)


def compare(section, name, example, quantity, value, cls="mean", show=True, check=False):
    # Relative difference from aggregate's number, which the CSV holds under (example, quantity).
    # check=True marks a consistency check: a row that holds by construction, not by agreement.
    agg = AGG[(example, quantity)]
    diff = abs(value - agg) / abs(agg) if agg else abs(value)
    _record(section, name, agg, value, diff, cls, show, check)


def compare_quantile(section, name, example, quantity, value, bs, cls="quantile", show=True):
    # Difference of a quantile in buckets.
    agg = AGG[(example, quantity)]
    _record(section, name, agg, value, abs(value - agg) / bs, cls, show, False)


def compare_distribution(section, name, example, prefix, grid, bs, share=False,
                         shown=("mean", "sd", "cv", "skew", "q0.5", "q0.9", "q0.99", "q0.999")):
    # share=True: a share placed off the grid, which aggregate re-buckets.
    # Skewness is compared with the skewness of aggregate's density (`_skew_central`); the skew
    # aggregate quotes (`_skew`) also counts the density's rounding deficit (see the last section).
    mean, sd, cv, skew = moments(grid)
    sd_cls = "share sd" if share else "sd"
    for stat, quantity, value, cls in [("mean", "mean", mean, "mean"), ("sd", "sd", sd, sd_cls),
                                       ("cv", "cv", cv, sd_cls), ("skew", "skew_central", skew, "skew")]:
        compare(section, f"{name} {stat}", example, f"{prefix}_{quantity}", value, cls, stat in shown)
    for p in (0.5, 0.9, 0.99, 0.999):
        compare_quantile(section, f"{name} q{p}", example, f"{prefix}_q{p}", grid.quantile(p), bs,
                         "share quantile" if share else "quantile", f"q{p}" in shown)


def report():
    # Print the comparisons made since the last report; * marks a consistency check.
    print(f"{'quantity':40s} {'aggregate':>14s} {'prospicio':>14s} {'difference':>12s}  pass")
    for r in SHOWN:
        diff = f"{r['diff']:.1e}" if r["unit"] == "rel" else f"{r['diff']:.2f} bkt"
        name = r["name"] + (" *" if r["check"] else "")
        print(f"{name:40s} {r['aggregate']:14.8g} {r['prospicio']:14.8g} {diff:>12s}  "
              f"{'yes' if r['ok'] else 'NO'}")
    SHOWN.clear()

## 1. A gross book and an occurrence tower

**The contract.** A book of ground-up claims, Poisson with 10 a year, each lognormal with mean 100 and CV 5 and limited by the policy to 5,000. A tower of five layers, each placed at 100% and per occurrence with no annual terms: 250 xs 0, 250 xs 250, 500 xs 500, 1,000 xs 1,000 and 3,000 xs 2,000. The tower covers the whole policy limit, so the layers add up to the gross loss.

In `aggregate` this is

```
agg Re.T 10 claims 5000 xs 0 sev lognorm 100 cv 5
    occurrence ceded to 250 xs 0 and 250 xs 250 and 500 xs 500 and 1000 xs 1000 and 3000 xs 2000 poisson
```

built with `normalize=False`. It picks the bucket `bs = 4` itself. The bucket does not divide 250, where 250 xs 0 detaches and 250 xs 250 attaches, so those two layers are split linearly between the two neighbouring buckets; `aggregate` does the same. The split keeps each layer consistent with the discretized severity, so the five layers still add up to the gross loss on the grid. The other boundaries (500, 1,000, 2,000 and 5,000) are grid points.

**In prospicio.** `discretize` rounds the severity to the grid, `fft` compounds it into the gross annual loss, and `Tower.on_grid` gives each layer's annual ceded loss from the same severity grid and claim count.

In [ ]:
S1 = "1 tower"
BS1 = 4.0
claims = Poisson(10.0)
severity = Lognormal.from_mean_cv(100.0, 5.0)
sev_grid = discretize(severity, BS1, 5000.0)
gross, gross_report = fft(claims, sev_grid, POINTS)

print(f"mean claim: exact {severity.lev(5000.0):.4f}, on the grid {sev_grid.mean():.4f}")
print(f"annual gross: exact {10 * severity.lev(5000.0):.4f}, on the grid {gross.mean():.4f}")
size = recommend_grid(claims, severity)
print(f"recommend_grid on the unlimited lognormal: step {size.step:g}; aggregate, which sees the 5,000 limit, "
      f"used {AGG[('tower_gross', 'bs')]:g}")
print(f"aliasing error {gross_report.aliasing_error:.1e}, mass beyond the grid {gross_report.tail_mass:.1e}")

compare(S1, "mean claim, exact", "tower_gross", "sev_mean_exact", severity.lev(5000.0))
compare(S1, "mean claim, grid", "tower_gross", "sev_mean_grid", sev_grid.mean())
compare(S1, "annual gross mean, exact", "tower_gross", "gross_mean_exact", 10 * severity.lev(5000.0))
compare_distribution(S1, "gross", "tower_gross", "gross", gross, BS1)
report()

The gross loss agrees with `aggregate` to rounding. The mean on the grid (950.16) is 0.044% below the exact mean (950.58) because rounding to buckets of 4 moves each claim by a bit; `aggregate` has the same gap, so the right comparison is grid against grid, which is what the table does. The exact row is a check of the closed form (`LEV`) on both sides.

**Choosing the bucket.** The bucket is passed by hand here, and in every section, only so that the grids match `aggregate`'s. In your own work let `fft_auto` choose it: it calls `recommend_grid` and compounds on the grid it picks. Neither can see a policy limit, because there is no limited severity to give them, so on this book they size the grid for the unlimited lognormal and pick 2 where `aggregate`, which sees the 5,000 limit, picks 4. The finer step is the safer side; it moves the grid numbers by discretisation, away from `aggregate`'s.

Now the tower. `TOWER` lists limit and attachment; `tower_grids.ceded` holds each layer's annual ceded distribution.

In [ ]:
TOWER = [(250.0, 0.0), (250.0, 250.0), (500.0, 500.0), (1000.0, 1000.0), (3000.0, 2000.0)]
tower = Tower([Layer(f"{lim:g} xs {att:g}", lim, att) for lim, att in TOWER])
tower_grids = tower.on_grid(claims, sev_grid, POINTS)

print(f"{'layer':>13s} {'mean':>9s} {'cv':>7s} {'loss on line':>13s} {'P(claim reaches)':>17s} "
      f"{'mean | reaches':>15s}")
for (lim, att), layer_grid in zip(TOWER, tower_grids.ceded):
    name = f"{lim:g} xs {att:g}"
    tag = f"layer_{lim:g}x{att:g}"  # the reference file's name
    ex = "tower_gross"
    compare_distribution(S1, f"layer {name}", ex, f"{tag}_ceded", layer_grid, BS1, shown=("mean",))
    compare(S1, f"layer {name} mean, exact", ex, f"{tag}_ceded_mean_exact", 10 * severity.layer(lim, att),
            show=False)
    reaches = 1.0 - sev_grid.cdf(att)
    cond = sev_grid.layer(lim, att) / reaches
    compare(S1, f"layer {name} loss on line", ex, f"{tag}_loss_on_line", layer_grid.mean() / lim, show=False)
    compare(S1, f"layer {name} P(claim reaches layer)", ex, f"{tag}_pr_attach_grid", reaches, show=False)
    compare(S1, f"layer {name} P(claim exhausts layer)", ex, f"{tag}_pr_detach_grid",
            prob_at_least(sev_grid, att + lim), show=False)
    compare(S1, f"layer {name} mean given it reaches", ex, f"{tag}_cond_sev_mean", cond, show=False)
    print(f"{name:>13s} {layer_grid.mean():9.3f} {moments(layer_grid)[2]:7.3f} {layer_grid.mean() / lim:13.4f} "
          f"{reaches:17.4f} {cond:15.3f}")

print(f"claims below half a bucket, which round to 0: {severity.cdf(BS1 / 2):.4f}; exact mean claim in 250 xs 0: "
      f"{severity.layer(250.0, 0.0):.3f}")
total = sum(c.mean() for c in tower_grids.ceded)
compare(S1, "sum of layers equals gross", "tower_gross", "tower_ceded_total_mean", total)
print(f"layers add up to {total:.4f}, gross is {gross.mean():.4f}; on grid points: {tower_grids.on_points}")
report()

Every layer's mean, standard deviation, skewness, quantiles, loss on line, attachment and exhaustion probability and conditional severity match `aggregate`'s single-layer builds (the table above shows the means; all of the others are in the summary at the end). The "mean | reaches" column is conditional on the claim reaching the layer, as in `aggregate`'s `reins_stats_df`; the loss on line is the unconditional annual mean over the limit.

The first row reads as if a tenth of the claims never reach a layer that attaches at 0. That is the grid: the lognormal puts 10.3% of claims below 2, half a bucket, and rounding sends them to 0, where they cede nothing. The 60.7 in the last column is therefore the mean over the 89.7% of claims that survive rounding; over all claims the exact mean claim in the layer is 54.5. `aggregate` reports the same 89.7% and 60.7.

`Tower.on_grid` says `on_points` is `False` because the bucket of 4 does not divide 250, where 250 xs 0 detaches and 250 xs 250 attaches. It still agrees with `aggregate`, which splits the same way; the flag warns that the shape of those two layers is smeared by up to a bucket, not their means.

## 2. Ceded and net of a program, and a share

**The contract.** The same book, now with a program of two layers, 500 xs 500 and 1,000 xs 1,000, both at 100% and per occurrence. The reinsurer pays the loss in those layers; the cedant retains the rest. Then a different placement: an 80% share of 1,000 xs 1,000.

In `aggregate`:

```
agg Re.T 10 claims 5000 xs 0 sev lognorm 100 cv 5 occurrence net of 500 xs 500 and 1000 xs 1000 poisson
agg Re.T 10 claims 5000 xs 0 sev lognorm 100 cv 5 occurrence ceded to 80% po 1000 xs 1000 poisson
```

**In prospicio.** `Tower.on_grid` gives the net annual loss and each layer's ceded loss. The two layers are contiguous, so the annual loss ceded to the whole program is that of the single layer 1,500 xs 500, which `Tower.on_grid` also gives. A program with a gap would need the severity grid mapped through its layers and compounded; the cell does that too and checks that the two agree. The share is `Layer(share=0.8)`: its ceded grid has step `0.8 * 4 = 3.2`, so the share adds no rounding of its own (the severity underneath is still rounded to 4), where `aggregate` re-buckets it onto the grid of 4.

In [ ]:
S2 = "2 program"
PROGRAM = [(500.0, 500.0), (1000.0, 1000.0)]
program = Tower([Layer(f"{lim:g} xs {att:g}", lim, att) for lim, att in PROGRAM])
program_grids = program.on_grid(claims, sev_grid, POINTS)
net = program_grids.net

# The two layers are contiguous, so the whole program is the single layer 1,500 xs 500.
program_ceded = Tower([Layer("1500 xs 500", 1500.0, 500.0)]).on_grid(claims, sev_grid, POINTS).ceded[0]
# The way a program with a gap is built: map the severity through its layers, then compound.
ceded_severity, _ = sev_grid.map(
    lambda x: sum(min(max(x - att, 0.0), lim) for lim, att in PROGRAM))
mapped, _ = fft(claims, ceded_severity, POINTS)
print(f"single layer against the mapped severity: largest difference in probability "
      f"{max(abs(a - b) for a, b in zip(program_ceded.probs, mapped.probs)):.1e}\n")

exact_ceded = 10 * sum(severity.layer(lim, att) for lim, att in PROGRAM)
print(f"{'':8s} {'mean':>10s} {'exact mean':>11s} {'sd':>9s} {'cv':>7s} {'99% quantile':>13s}")
for label, grid, exact in [("gross", gross, 10 * severity.lev(5000.0)), ("ceded", program_ceded, exact_ceded),
                           ("net", net, 10 * severity.lev(5000.0) - exact_ceded)]:
    mean, sd, cv, _ = moments(grid)
    print(f"{label:8s} {mean:10.3f} {exact:11.3f} {sd:9.1f} {cv:7.3f} {grid.quantile(0.99):13.1f}")

compare_distribution(S2, "gross", "tower_program", "gross", gross, BS1, shown=("mean", "q0.99"))
compare_distribution(S2, "ceded", "tower_program", "ceded", program_ceded, BS1)
compare_distribution(S2, "net", "tower_program", "net", net, BS1)
compare(S2, "ceded mean, exact", "tower_program", "ceded_mean_exact", exact_ceded)
compare(S2, "net mean, exact", "tower_program", "net_mean_exact", 10 * severity.lev(5000.0) - exact_ceded)
compare(S2, "net mean claim, grid", "tower_program", "net_sev_mean", sev_grid.mean() - ceded_severity.mean())
for (lim, att), layer_grid in zip(PROGRAM, program_grids.ceded):
    tag = f"layer_{lim:g}x{att:g}"
    compare_distribution(S2, f"layer {lim:g} xs {att:g}", "tower_program", f"{tag}_ceded", layer_grid, BS1,
                         shown=("mean", "cv"))
report()

Gross, ceded and net all match, including the net view that `aggregate` holds in `reins_density_df`. Ceded and net add up in the mean (950.16 = 201.90 + 748.26) but not in the standard deviation: 456.1 + 640.7 = 1,096.8 is more than the gross 1,016.6. Ceded and net come from the same claims, so they move together, but not in lockstep (a year of many small claims is all net), and the gross spread is less than the sum of the two. That diversification is what a reinsurer cannot see from inside its own layer.

Now the 80% share of 1,000 xs 1,000.

In [ ]:
share_layer = Layer("80% of 1000 xs 1000", 1000.0, 1000.0, share=0.8)
share_grids = Tower([share_layer]).on_grid(claims, sev_grid, POINTS)
share_ceded = share_grids.ceded[0]
print(f"step of the ceded grid {share_ceded.step:g}, mean {share_ceded.mean():.4f}, "
      f"exact {0.8 * 10 * severity.layer(1000.0, 1000.0):.4f}")

compare_distribution(S2, "80% of 1000 xs 1000", "tower_program", "share80_1000x1000_ceded", share_ceded, BS1,
                     share=True)
compare(S2, "80% of 1000 xs 1000 mean, exact", "tower_program", "share80_1000x1000_ceded_mean_exact",
        0.8 * 10 * severity.layer(1000.0, 1000.0), show=False)
report()

The mean agrees to rounding. The standard deviation differs by 2.6e-6, the skewness by 1.5e-6, and the 90% quantile by 0.8, a fifth of a bucket: `aggregate` spreads each ceded amount that falls between its buckets of 4 over the two neighbours, which adds a little variance, where `prospicio` keeps the ceded grid at step 3.2. The standard deviation and quantiles of a share placed off the grid therefore have their own tolerances (1e-5 and one bucket), and the difference is the one that `aggregate`'s re-bucketing introduces.

The program's gross, ceded and net annual distributions:

In [ ]:
import matplotlib.pyplot as plt

xs = [i * 40.0 for i in range(201)]
fig, ax = plt.subplots(figsize=(7, 3.8))
for label, grid in [("gross", gross), ("net of the program", net), ("ceded to the program", program_ceded)]:
    ax.plot(xs, [1.0 - grid.cdf(x) for x in xs], label=label)
ax.set_yscale("log")
ax.set_ylim(1e-4, 1.2)
ax.set_xlabel("annual loss (currency units)")
ax.set_ylabel("probability of exceeding")
ax.set_title("Exceedance curves, 10 claims, 500 xs 500 and 1,000 xs 1,000")
ax.legend()
fig.tight_layout()
plt.show()

## 3. An annual aggregate limit and reinstatements

**The contract.** Bear and Nemlick's Treaty 2: three classes of business with subject premium 2,000 each and expected layer loss 10%, 14% and 21% of it. The layer is 700 xs 300 of a single-parameter Pareto with shape 1.5, 1.3 and 1.1, which is a Lomax of that shape, scale 300, limited to 700. The claim count is gamma-mixed Poisson with mixing CV 0.07, one mixing variable shared by all classes. The treaty has an annual aggregate limit of 2,800, which is three reinstatements plus the original 700.

```
agg Re.BN2 [2000 2000 2000] exposure at [.1 .14 .21] rate 700 xs 0
    sev 300 * pareto [1.5 1.3 1.1] - 300 mixed gamma 0.07              (bs = 1/8)
agg Re.BN2 ... aggregate ceded to 2800 xs 0
```

`exposure at rate` fixes the expected loss of each class, and the claim count is solved from it. `exposure_book` does the same with `Burr(shape, 1, scale)` (the Lomax) and its limited expected value; the three counts are a check on that step. With one shared mixing variable the three classes are one mixed Poisson over a mixture of the three severities, so a single `fft` gives the gross annual loss.

**Reinstatements.** The `aggregate` docs have no reinstatement example, so the schedules here are designed: three reinstatements at 100% (A), and the first free, the second at 50% and the third at 100% (B). They are pro rata as to amount, the only kind `aggregate` has: in a year whose layer loss is `L`, the k-th reinstatement restores the k-th 700 of limit used up, and its premium is `deposit x rate_k x min(max(L - (k - 1) 700, 0), 700) / 700`, summed over k = 1, 2, 3. `aggregate` computes it with its own `ReinstatementTerms` on the annual layer loss.

The deposit of 1,200 (a rate on line of 171%) is a designed input to the comparison, not a price: it is not set from the loss cost. Because the reinstatement premium is proportional to the deposit, a price comes from solving for the deposit that gives a target loss ratio, which the cell after the comparison does for each schedule.

In [ ]:
S3 = "3 limit and reinstatements"
BS3 = 1 / 8
counts3, n3, mix3 = exposure_book([2000.0] * 3, [0.10, 0.14, 0.21], [1.5, 1.3, 1.1], 300.0, 700.0)
freq3 = mixed_poisson(n3, 0.07)
sev3 = discretize(mix3, BS3, 700.0)
gross3, report3 = fft(freq3, sev3, POINTS)
print(f"expected claims by class {[round(c, 4) for c in counts3]}, total {n3:.4f}")
print(f"frequency {freq3}")
print(f"mass beyond the grid {report3.tail_mass:.1e}")

for i, c in enumerate(counts3, 1):
    compare(S3, f"class {i} expected claims", "agg_limit_reinst", f"class{i}_expected_count", c)
compare(S3, "expected claims", "agg_limit_reinst", "claim_count_mean", n3)
compare(S3, "gross mean, exact", "agg_limit_reinst", "gross_mean_exact", n3 * mix3.lev(700.0))
exact_cv3 = math.sqrt(n3 * mix3.layer_second_moment(700.0, 0.0) + 0.07**2 * (n3 * mix3.lev(700.0)) ** 2) \
    / (n3 * mix3.lev(700.0))
compare(S3, "gross CV, exact", "agg_limit_reinst", "gross_cv_exact", exact_cv3)
compare_distribution(S3, "gross", "agg_limit_reinst", "gross", gross3, BS3)
report()

In [ ]:
limit_layer = Layer("2800 aggregate limit", 700.0, 0.0, aggregate_limit=2800.0)
limited = Tower([limit_layer]).on_grid(freq3, sev3, POINTS).ceded[0]
beyond = gross3.stop_loss(2800.0)
print(f"mean recovery {money(limited.mean())}, P(limit used up) {prob_at_least(gross3, 2800.0):.5f}, "
      f"loss above the limit {beyond:.4f}")

compare_distribution(S3, "recovery under 2800", "agg_limit_reinst", "agglimit2800_ceded", limited, BS3,
                     shown=("mean", "sd", "q0.99"))
compare(S3, "P(limit used up)", "agg_limit_reinst", "agglimit2800_pr_hit", prob_at_least(gross3, 2800.0))
compare(S3, "loss above the limit", "agg_limit_reinst", "agglimit2800_mean_loss_beyond", beyond, "tail")
report()

In [ ]:
DEPOSIT, LIMIT3, TARGET_LR = 1200.0, 700.0, 0.65
per_deposit = {}
for tag, rates in [("A", [1.0, 1.0, 1.0]), ("B", [0.0, 0.5, 1.0])]:
    layer = Layer(f"schedule {tag}", LIMIT3, 0.0, reinstatement_rates=rates).with_deposit_premium(DEPOSIT)
    grids = Tower([layer]).on_grid(freq3, sev3, POINTS)
    reinstatement = grids.expected_reinstatement_premium[0]
    recovery = grids.ceded[0].mean()
    total = layer.premium + reinstatement
    per_deposit[tag] = (reinstatement / layer.premium, recovery)
    print(f"schedule {tag} {rates}: deposit {money(layer.premium)} (rate on line {layer.premium / LIMIT3:.0%}), "
          f"expected reinstatement premium {money(reinstatement)}, total {money(total)}")
    print(f"    expected recovery {money(recovery)}, loss ratio {recovery / total:.1%}, "
          f"margin {money(total - recovery)}")
    compare(S3, f"{tag}: expected reinstatement premium", "agg_limit_reinst", f"reinst{tag}_premium_mean",
            reinstatement)
    compare(S3, f"{tag}: total premium", "agg_limit_reinst", f"reinst{tag}_total_premium_mean", total)
    compare(S3, f"{tag}: recovery", "agg_limit_reinst", f"reinst{tag}_recovery_mean", recovery)
    compare(S3, f"{tag}: P(any loss to the layer)", "agg_limit_reinst", f"reinst{tag}_pr_any_reinstatement",
            1.0 - gross3.probs[0])
    compare(S3, f"{tag}: maximum premium", "agg_limit_reinst", f"reinst{tag}_max_premium",
            layer.premium * sum(rates), show=False, check=True)
report()

# The deposit that prices each schedule at the target loss ratio. The total premium is the deposit
# times (1 + k), where k is the expected reinstatement premium per unit of deposit.
print()
for tag, (k, recovery) in per_deposit.items():
    deposit = recovery / TARGET_LR / (1 + k)
    print(f"schedule {tag}: a deposit of {money(deposit)} (rate on line {deposit / LIMIT3:.0%}) prices it at a "
          f"{TARGET_LR:.0%} loss ratio, total {money(deposit * (1 + k))}, {k / (1 + k):.0%} of it from reinstatements")

The expected reinstatement premium and the recovery agree to 1e-8 or better. The recovery under schedule A or B is the same 894.68, because both have the same annual limit of 2,800, and it is the mean of the aggregate-limit layer above. `aggregate`'s probability of a reinstatement is the probability of any loss to the layer, `P(L > 0)`; under schedule B the first reinstatement is free, so a reinstatement premium is due only when `L` passes 700.

At the designed deposit, schedule A collects 2,695.57 against an expected recovery of 894.68 (a 33% loss ratio) and B collects 1,571.58 (57%). Priced to the same 65% loss ratio, both collect 1,376.43 in expectation, but A needs the smaller deposit: it charges for every reinstatement, so more of its premium arrives after losses and less is paid up front. The choice of schedule moves premium between the deposit and the loss-contingent part, not the expected total.

The one looser tolerance in this section is the stop loss at 2,800, deep in the tail (the loss above the limit). It reads the far tail, and `prospicio` keeps the 6e-10 of probability beyond the grid's top on its last point where `aggregate` cuts it; that shifts the result by 6e-7 relative, inside the 1e-5 set for it.

The gross annual loss and the recovery under the limit (the steps at multiples of 700 are probability mass from claims that hit the 700 limit):

In [ ]:
import matplotlib.pyplot as plt

xs = [i * 10.0 for i in range(421)]
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(xs, [1.0 - gross3.cdf(x) for x in xs], label="gross annual layer loss")
ax.plot(xs, [1.0 - limited.cdf(x) for x in xs], color="C3", label="recovery under the 2,800 limit")
ax.axvline(2800.0, color="grey", ls="--", lw=1)
ax.set_yscale("log")
ax.set_ylim(1e-4, 1.2)
ax.set_xlabel("annual loss (currency units)")
ax.set_ylabel("probability of exceeding")
ax.set_title("Treaty 2: the limit cuts the tail at 2,800")
ax.legend()
fig.tight_layout()
plt.show()

## 4. Aggregate stop loss and aggregate deductible

**Stop loss.** On the book of section 1, a cover on the annual total: 250 xs 750, 1,500 xs 1,000, the two together (750 to 2,500) and a 50% share of 1,500 xs 1,000. `Layer.stop_loss(name, limit, retention)` builds the three covers at 100%. It takes no share, so the 50% share is the `Layer` constructor with `share=0.5`, no per-occurrence limit, and the retention and limit as annual terms. The book has no premium, so the covers are quoted in amounts. In the market a stop loss is usually quoted in loss ratio points of subject premium, and the amounts are the points times the premium: on a subject premium of 1,000 (an expected loss ratio of 95%), 250 xs 750 would be 25% xs 75%.

```
agg Re.T 10 claims 5000 xs 0 sev lognorm 100 cv 5 poisson aggregate ceded to 250 xs 750
agg Re.T ...                                    poisson aggregate ceded to 50% po 1500 xs 1000
```

**Aggregate deductible.** Bear and Nemlick's Treaty 1: the 160 xs 40 layer of a single-parameter Pareto with shape 0.9 and 0.95 on two classes (subject premiums 9,000 and 3,000, expected layer loss 4% and 3% of them, 450 in all), gamma-mixed Poisson with the mixing CV that gives the CV the text quotes, `((1.036 - 1) / 5.154)**0.5`. The reinsurer pays above an annual aggregate deductible of 360 (3% of 12,000), `aggregate net of 360 xs 0` in `aggregate`. The guide's table gives the subject premiums as 3,000 and 6,000, but its code uses 9,000 and 3,000 at 4% and 3%, which gives the 450 (3.75% of 12,000) that its Treaty 4 text states; the notebook follows the code.

In [ ]:
S4 = "4 stop loss"


def annual_cover(layer):
    # The annual ceded and net of one aggregate cover on the book of section 1.
    return Tower([layer]).on_grid(claims, sev_grid, POINTS)


for lim, att in [(250.0, 750.0), (1500.0, 1000.0)]:
    name = f"{lim:g} xs {att:g}"
    cover = annual_cover(Layer.stop_loss(name, lim, att))
    tag = f"asl_{lim:g}x{att:g}"
    compare_distribution(S4, name, "stop_loss", f"{tag}_ceded", cover.ceded[0], BS1,
                         shown=("mean", "cv", "q0.9"))
    compare(S4, f"{name} P(attach)", "stop_loss", f"{tag}_pr_attach", 1.0 - gross.cdf(att))
    compare(S4, f"{name} P(exhaust)", "stop_loss", f"{tag}_pr_exhaust", prob_at_least(gross, att + lim))
    compare(S4, f"{name} loss on line", "stop_loss", f"{tag}_loss_on_line", cover.ceded[0].mean() / lim)

both = annual_cover(Layer.stop_loss("750 to 2500", 1750.0, 750.0))
# Layer.stop_loss takes no share: a share of a stop loss is a layer with no per-occurrence limit
# and the retention and limit as annual terms.
half = annual_cover(Layer("50% of 1500 xs 1000", math.inf, 0.0, share=0.5, aggregate_deductible=1000.0,
                          aggregate_limit=1500.0))
compare_distribution(S4, "both layers, ceded", "stop_loss", "asl_both_ceded", both.ceded[0], BS1,
                     shown=("mean", "cv"))
compare_distribution(S4, "both layers, net", "stop_loss", "asl_both_net", both.net, BS1,
                     shown=("mean", "cv", "q0.9"))
compare_distribution(S4, "50% of 1500 xs 1000", "stop_loss", "asl_50pct_1500x1000_ceded", half.ceded[0], BS1,
                     share=True, shown=("mean", "cv"))
print(f"750 to 2,500: ceded {both.ceded[0].mean():.3f} + net {both.net.mean():.3f} = {gross.mean():.3f}")
report()

The stop-loss covers agree on every statistic. The 50% share has the same off-grid explanation as the 80% share in section 2.

The aggregate deductible:

In [ ]:
S4B = "4 aggregate deductible"
BS4 = 0.0625
MIX_CV4 = ((1.036 - 1) / 5.154) ** 0.5
counts4, n4, mix4 = exposure_book([9000.0, 3000.0], [0.04, 0.03], [0.9, 0.95], 40.0, 160.0)
sev4 = discretize(mix4, BS4, 160.0)
aad = Layer("AAD 360", math.inf, 0.0, aggregate_deductible=360.0)

gross4, _ = fft(mixed_poisson(n4, MIX_CV4), sev4, POINTS)
after_aad = Tower([aad]).on_grid(mixed_poisson(n4, MIX_CV4), sev4, POINTS).ceded[0]
gross4_poisson, _ = fft(Poisson(n4), sev4, POINTS)
after_aad_poisson = Tower([aad]).on_grid(Poisson(n4), sev4, POINTS).ceded[0]

absorbed = gross4.lev(360.0)
print(f"expected claims {n4:.4f}, gross CV {moments(gross4)[2]:.4f} (the text quotes 0.528)")
print(f"expected loss {money(gross4.mean())}; after the deductible {money(after_aad.mean())}; "
      f"absorbed by the deductible {money(absorbed)}")
print(f"loss elimination ratio (the cedant's credit) {absorbed / gross4.mean():.4f}; excess ratio (the reinsurer's "
      f"share, aggregate's insurance charge) {after_aad.mean() / gross4.mean():.4f}")
print(f"with Poisson counts the loss after the deductible falls to {money(after_aad_poisson.mean())}")

for i, c in enumerate(counts4, 1):
    compare(S4B, f"class {i} expected claims", "stop_loss_aad", f"class{i}_expected_count", c, show=False)
compare(S4B, "gross mean, exact", "stop_loss_aad", "gross_mean_exact", n4 * mix4.lev(160.0))
exact_cv4 = math.sqrt(n4 * mix4.layer_second_moment(160.0, 0.0) + MIX_CV4**2 * (n4 * mix4.lev(160.0)) ** 2) \
    / (n4 * mix4.lev(160.0))
compare(S4B, "gross CV, exact", "stop_loss_aad", "gross_cv_exact", exact_cv4)
compare_distribution(S4B, "gross", "stop_loss_aad", "gross", gross4, BS4, shown=("mean", "cv", "q0.99"))
compare_distribution(S4B, "after the deductible", "stop_loss_aad", "after_aad", after_aad, BS4)
compare(S4B, "value of the deductible", "stop_loss_aad", "aad_value", absorbed)
compare(S4B, "excess ratio (insurance charge)", "stop_loss_aad", "aad_insurance_charge",
        after_aad.mean() / gross4.mean())
compare_distribution(S4B, "Poisson, after the deductible", "stop_loss_aad", "poisson_after_aad",
                     after_aad_poisson, BS4, shown=("mean", "cv"))
compare(S4B, "Poisson gross mean", "stop_loss_aad", "poisson_gross_mean", gross4_poisson.mean(), show=False)
report()

`aggregate net of 360 xs 0` removes the first 360 of each year's loss (the cedant's deductible), so its density is the reinsurer's loss above 360; `Layer(aggregate_deductible=360)` is that side too. The deductible's value to the cedant is `E[min(X, 360)]`, `Grid.lev(360)`: 307.24, a loss elimination ratio of 68.3% of the 450 expected loss. The other 31.7%, the excess ratio, reaches the reinsurer. `aggregate` calls that ratio the insurance charge, which is the Table M charge at an entry ratio of 360 / 450 = 0.8.

## 5. Retro and swing premium, profit commission and sliding scale

Four contracts whose price depends on the loss. In `aggregate` each is a map `phi` of the year's loss, from `aggregate.contract_terms`, applied to the density; in `prospicio` they are terms of the `Layer` that `Tower.on_grid` averages over the annual distribution (`expected_premium`, `expected_profit_commission`, `expected_ceding_commission`).

**Retro rating** (Treaty 4). The Treaty 1 book with Poisson counts, the whole 160 xs 40 layer at 100%. Subject premium 12,000, loss conversion factor 100/75, minimum rate 3% (360), maximum rate 10% (1,200): premium `clip(lcf x loss, 360, 1200)`. Retro terms are quoted as rates on subject premium and the loss as an amount; the map works in amounts, so each rate is multiplied by the 12,000. That is a swing premium with no basic premium, so `with_swing_rating(0, lcf, 360, 1200)`; `LossSensitivePremium` is the same map as a stand-alone object.

**Swing rating** (designed). The same loss: premium `clip(300 + 1.25 loss, 360, 1200)`, at 100% and at a 40% placed share. The terms are quoted for 100%. `prospicio` scales the basic premium, the minimum and the maximum by the share, as `aggregate`'s underwriter does, and not the loss conversion factor.

**Profit commission** (Treaty 5). Three independent years of the Treaty 2 book with Poisson counts, so the sum is a compound Poisson with three times the claim count. The reinsurer takes 80% of the layer for a ceded premium of 4,500 (25% of 18,000). Profit commission is 25% of what is left after a 20% allowance and the losses, `0.25 max(0, 1 - 0.2 - LR)`, with the loss ratio on the ceded premium. This is the one-period form: the commission is settled once on the three-year total, with no deficit carried forward from a bad year, and the years are independent (Treaty 2's mixing variable is dropped). A deficit carry-forward or a mixing variable shared across the years would make the commission more variable and change the gap between its mean and its value at the expected loss ratio.

**Sliding scale** (Treaty 6). A 900 xs 100 layer of a Pareto with shape 1.05 on 25,000 of subject premium (expected loss 2,500, gamma mixing CV 0.095), ceded premium 5,000. The commission is 40% at or below a 35% loss ratio, 25% at 55%, 20% at 65%, linear between and flat outside.

In [ ]:
S5 = "5 retro and swing"
BS5 = 0.0625
counts5, n5, mix5 = exposure_book([9000.0, 3000.0], [0.04, 0.03], [0.9, 0.95], 40.0, 160.0)
sev5 = discretize(mix5, BS5, 160.0)
LCF = 100 / 75

retro = Layer("retro", math.inf, 0.0).with_swing_rating(0.0, LCF, 360.0, 1200.0)
retro_grids = Tower([retro]).on_grid(Poisson(n5), sev5, POINTS)
layer_loss = retro_grids.ceded[0]

# The same premium from the stand-alone map, averaged by hand over the annual distribution.
terms = LossSensitivePremium(0.0, LCF, 360.0, 1200.0)
points = [i * layer_loss.step for i in range(len(layer_loss.probs))]
by_hand = sum(p * q for p, q in zip(terms.premium(points), layer_loss.probs))
at_min = sum(q for x, q in zip(points, layer_loss.probs) if LCF * x <= 360.0)
at_max = sum(q for x, q in zip(points, layer_loss.probs) if LCF * x >= 1200.0)

print(f"expected layer loss {money(layer_loss.mean())}, expected retro premium "
      f"{money(retro_grids.expected_premium[0])} ({retro_grids.expected_premium[0] / 12000:.3%} of 12,000)")
print(f"P(minimum premium) {at_min:.4f}, P(maximum premium) {at_max:.4f}")

compare_distribution(S5, "layer loss", "variable_retro", "ceded_loss", layer_loss, BS5,
                     shown=("mean", "cv", "q0.99"))
compare(S5, "retro premium", "variable_retro", "retro_premium_mean", retro_grids.expected_premium[0])
compare(S5, "retro premium, LossSensitivePremium", "variable_retro", "retro_premium_mean_via_terms", by_hand)
compare(S5, "retro rate on subject premium", "variable_retro", "retro_rate_mean",
        retro_grids.expected_premium[0] / 12000.0)
compare(S5, "P(minimum premium)", "variable_retro", "pr_min_binds", at_min)
compare(S5, "P(maximum premium)", "variable_retro", "pr_max_binds", at_max)

for share, tag in [(1.0, "share100"), (0.4, "share40")]:
    swing = Layer("swing", math.inf, 0.0, share=share).with_swing_rating(300.0, 1.25, 360.0, 1200.0)
    premium = Tower([swing]).on_grid(Poisson(n5), sev5, POINTS).expected_premium[0]
    print(f"swing premium at {share:.0%} placed: {money(premium)}")
    compare(S5, f"swing premium, {share:.0%} placed", "variable_swing", f"{tag}_premium_mean", premium)
report()

In [ ]:
S5B = "5 profit commission"
BS5B = 0.25
counts5b, n5b, mix5b = exposure_book([2000.0] * 3, [0.10, 0.14, 0.21], [1.5, 1.3, 1.1], 300.0, 700.0)
sev5b = discretize(mix5b, BS5B, 700.0)
three_years = Poisson(3 * n5b)  # three independent years of Poisson counts

PREMIUM5B = 0.25 * 18000.0
pc_layer = (Layer("treaty 5", 700.0, 0.0, share=0.8, premium=PREMIUM5B)
            .with_profit_commission(0.25, allowance=0.2))
pc_grids = Tower([pc_layer]).on_grid(three_years, sev5b, POINTS)
total_loss, _ = fft(three_years, sev5b, POINTS)
lr = pc_grids.ceded[0].mean() / PREMIUM5B
commission = pc_grids.expected_profit_commission[0]

print(f"three years: expected loss {money(total_loss.mean())}, CV {moments(total_loss)[2]:.4f}")
print(f"ceded loss ratio {lr:.4f}; expected profit commission {money(commission)} = "
      f"{commission / PREMIUM5B:.3%} of the ceded premium")
print(f"at the expected loss ratio alone it would be {pc_layer.profit_commission_for(pc_grids.ceded[0].mean()) / PREMIUM5B:.3%}")

compare_distribution(S5B, "three-year loss", "variable_pc", "three_year_loss", total_loss, BS5B,
                     shown=("mean", "cv", "q0.99"))
compare(S5B, "ceded loss ratio", "variable_pc", "ceded_lr_mean", lr)
compare(S5B, "profit commission", "variable_pc", "pc_amount_mean", commission)
compare(S5B, "profit commission rate", "variable_pc", "pc_rate_mean", commission / PREMIUM5B)
compare(S5B, "rate at the expected loss ratio", "variable_pc", "pc_at_expected_lr",
        pc_layer.profit_commission_for(pc_grids.ceded[0].mean()) / PREMIUM5B)
compare(S5B, "P(profit commission > 0)", "variable_pc", "pr_pc_positive",
        sum(q for i, q in enumerate(pc_grids.ceded[0].probs)
            if i * pc_grids.ceded[0].step / PREMIUM5B < 1 - 0.2))
report()

In [ ]:
S5C = "5 sliding scale"
BS5C = 0.5
counts5c, n5c, mix5c = exposure_book([25000.0], [0.10], [1.05], 100.0, 900.0)
sev5c = discretize(mix5c, BS5C, 900.0)
frequency5c = mixed_poisson(n5c, 0.095)

PREMIUM5C = 25000.0 * 0.20
ANCHORS = [(0.40, 0.35), (0.25, 0.55), (0.20, 0.65)]  # (commission, loss ratio)
slide = Layer("treaty 6", 900.0, 0.0, premium=PREMIUM5C).with_sliding_scale(ANCHORS)
slide_grids = Tower([slide]).on_grid(frequency5c, sev5c, POINTS)
slide_loss = slide_grids.ceded[0]
slide_commission = slide_grids.expected_ceding_commission[0]
at_expected = slide.ceding_commission_for(slide_loss.mean()) / PREMIUM5C
lr_probs = [(i * slide_loss.step / PREMIUM5C, q) for i, q in enumerate(slide_loss.probs)]

print(f"expected claims {n5c:.4f}; expected ceded loss {money(slide_loss.mean())}, loss ratio "
      f"{slide_loss.mean() / PREMIUM5C:.3f}")
print(f"expected commission {money(slide_commission)} = {slide_commission / PREMIUM5C:.2%} of premium")
print(f"commission at the expected loss ratio {at_expected:.2%}")


# The gap between the two comes from where the scale leaves its middle segment (slope -0.75),
# extended in a straight line: capped at 40% below a 35% loss ratio, flatter above 55%.
def scale(r):
    return slide.ceding_commission_for(r * PREMIUM5C) / PREMIUM5C


def middle(r):
    return 0.40 - 0.75 * (r - 0.35)


below = sum(q * (scale(x) - middle(x)) for x, q in lr_probs if x < 0.35)
above = sum(q * (scale(x) - middle(x)) for x, q in lr_probs if x > 0.55)
print(f"against the middle segment extended, years below a 35% loss ratio take {-below:.2%} off the expected "
      f"commission and years above 55% add {above:.2%}")

compare(S5C, "expected claims", "variable_slide", "claim_count_mean", n5c, show=False)
compare_distribution(S5C, "ceded loss", "variable_slide", "ceded_loss", slide_loss, BS5C,
                     shown=("mean", "cv", "q0.99"))
compare(S5C, "ceded loss ratio", "variable_slide", "ceded_lr_mean", slide_loss.mean() / PREMIUM5C)
compare(S5C, "expected commission", "variable_slide", "slide_commission_amount_mean", slide_commission)
compare(S5C, "expected commission rate", "variable_slide", "slide_commission_rate_mean",
        slide_commission / PREMIUM5C)
compare(S5C, "rate at the expected loss ratio", "variable_slide", "slide_commission_at_expected_lr", at_expected)
compare(S5C, "P(loss ratio <= 35%)", "variable_slide", "pr_lr_below_35", sum(q for x, q in lr_probs if x <= 0.35))
compare(S5C, "P(loss ratio >= 65%)", "variable_slide", "pr_lr_above_65", sum(q for x, q in lr_probs if x >= 0.65))
report()

All four agree. Three points of reading:

* A variable premium or commission is a function of the whole annual distribution, not of the expected loss, and the average of a non-linear map differs from the map at the average. The profit commission, `0.25 max(0, 0.8 - LR)`, is convex in the loss ratio, so its average (8.24% of premium) is above its value at the expected loss ratio (8.00%), as Jensen's inequality says it must be. The sliding scale is not convex: it bends down at 35% (where it stops at 40%) and up at 55% and 65%. Its average (30.3%) is above its value at the expected loss ratio (28.8%) only because this loss ratio is skewed to the right: the years above 55%, where the scale flattens, add more than the 29.5% of years below 35% take off (the cell above splits the gap). For a book with a different spread the gap could have the other sign. The `expected_*` results average the terms over the distribution, which a price from the expected loss alone misses.
* `aggregate`'s documentation for the sliding scale builds the same schedule in code with a slope of 0.25 on the first segment where its text says 0.75, which gives a commission of 35.7%, not 30.3%. The table compares the terms the text states, which are what `with_sliding_scale` takes.
* `aggregate`'s profit commission does not deduct a ceding commission; `with_profit_commission` follows it, and its allowance covers any commission the contract takes before profit.

The commission scale and the loss ratio it is applied to:

In [ ]:
import matplotlib.pyplot as plt

lrs = [i / 200 for i in range(0, 161)]
fig, ax = plt.subplots(figsize=(7, 3.8))
dx = slide_loss.step / PREMIUM5C
upto = int(0.8 / dx) + 1  # grid points up to a loss ratio of 80%
ax.fill_between([x for x, _ in lr_probs[:upto]], [q / dx for _, q in lr_probs[:upto]], color="C0", alpha=0.35,
                label="density of the loss ratio")
ax.set_xlim(0, 0.8)
ax.set_xlabel("ceded loss ratio (loss / 5,000 ceded premium)")
ax.set_ylabel("probability density")
ax2 = ax.twinx()
ax2.plot(lrs, [slide.ceding_commission_for(x * PREMIUM5C) / PREMIUM5C for x in lrs], color="C3", label="commission")
ax2.set_ylabel("commission (fraction of ceded premium)")
ax2.set_ylim(0, 0.5)
ax.set_title("Treaty 6: sliding-scale commission and the loss ratio")
fig.legend(loc="upper right", bbox_to_anchor=(0.88, 0.85))
fig.tight_layout()
plt.show()

## 6. Risk-adjusted premium with distortions

**The contract.** The program of section 2 priced as a risk, in two views: the reinsurer's ceded loss (the annual loss to 500 xs 500 and 1,000 xs 1,000) and the cedant's gross annual loss. The capital standard is assets `a` at the 99% quantile of the view; the cost of capital is 10%, so the premium `P` solves `P = L + 0.10 (a - P)` with `L = E[min(X, a)]`, the CCoC premium. It is a risk-adjusted loss cost: the expected loss plus the cost of the capital it ties up, before expenses, brokerage and ceding commission, and with no investment income. Each of the proportional hazard, Wang, dual power and TVaR distortions is then calibrated so that its price of `min(X, a)` equals `P`, and a second set of fixed-parameter distortions is priced at the same assets.

`aggregate` does this with `Aggregate.calibrate_distortions(0.10, p=0.99, reins_view=...)`. These examples are designed: its pricing pages hold no worked example for this book.

**In prospicio.** CCoC is itself a distortion, `g(s) = (r + s) / (1 + r)` for `s > 0`, so `Distortion.ccoc(0.10).measure` of `min(X, a)` is the premium, and `Grid.lev(a)` is `L`. `prospicio.risk.calibrate(family, dist, premium, assets=a)` finds the parameter whose price of `min(X, a)` is the premium; `Distortion.measure` prices a grid. `aggregate` takes `a` as the lower 99% quantile on the grid, and so does `Grid.quantile`, so `F(a)` is a little above 99%. `Distortion.family` and `Distortion.parameters` read the calibrated family and its parameter off the object, so each parameter is compared with the one `aggregate` calibrated. `calibrate` takes a family's short name (`ph`, `dual`), which is the table's first column, and `Distortion.family` returns the constructor's name (`proportional_hazard`, `dual_power`), which labels the calibrated distortion. The distortion function `g(s)` is compared too, at two survival probabilities, with `aggregate`'s parameter put into `aggregate`'s formula: that checks that the two packages define each family the same way, which the parameter alone cannot.

**Reading the table.** "price of min(X, a)" is the premium each calibrated distortion charges at the capital standard, so it equals `P` by construction. "unlimited" is the same distortion applied to `X` with no cap: the premium it would charge if the reinsurer's liability were not limited to the assets. The gap between the two is the load the distortion puts on the tail beyond the 99% standard. CCoC shows n/a, because its jump at `s = 0` charges the cost of capital on the largest possible loss, so the cap is part of its definition.

In [ ]:
S6 = "6 risk pricing"
normal = NormalDist()
AGG_G = {  # g(s) from the parameter aggregate calibrated
    "ccoc": lambda r, s: min(1.0, (r + s) / (1 + r)),
    "ph": lambda rho, s: s**rho,
    "wang": lambda lam, s: normal.cdf(normal.inv_cdf(s) + lam),
    "dual": lambda b, s: 1.0 - (1.0 - s) ** b,
    "tvar": lambda p, s: min(1.0, s / (1.0 - p)),
}
COC, CAPITAL_P = 0.10, 0.99
OCCURRENCE_LIMIT = 1500.0  # the program's limit per occurrence: 500 xs 500 and 1,000 xs 1,000
FIXED = [("ph0.7", Distortion.proportional_hazard(0.7)), ("wang0.4", Distortion.wang(0.4)),
         ("dual1.8", Distortion.dual_power(1.8)), ("tvar0.3", Distortion.tvar(0.3)),
         ("ccoc0.1", Distortion.ccoc(0.1))]
priced = {}

for view, example, x in [("ceded", "risk_pricing_ceded", program_ceded), ("gross", "risk_pricing_gross", gross)]:
    s = f"6 {view} view"
    assets = x.quantile(CAPITAL_P)
    capped, _ = x.map(lambda v: min(v, assets))
    loss = x.lev(assets)  # E[min(X, a)]
    premium = Distortion.ccoc(COC).measure(capped)  # (L + r a) / (1 + r)
    capital = assets - premium
    compare(s, "assets, the 99% quantile", example, "cal_a", assets)
    compare(s, "F(assets)", example, "cal_F(a)", x.cdf(assets))
    compare(s, "expected loss E[min(X, a)]", example, "cal_L", loss)
    compare(s, "CCoC premium", example, "cal_P", premium)
    # a - P, and (P - L) / (a - P) = r: they follow from the rows above.
    compare(s, "capital", example, "cal_Q", capital, check=True)
    compare(s, "return on capital", example, "cal_ROE", (premium - loss) / capital, check=True)
    report()

    print(f"{view}: assets {money(assets)}, expected loss {money(loss)}, premium {money(premium)}, "
          f"capital {money(capital)}")
    rate_on_line = f", rate on line {premium / OCCURRENCE_LIMIT:.1%}" if view == "ceded" else ""
    print(f"    loss ratio {loss / premium:.1%}, margin {money(premium - loss)} ({COC:.0%} of the capital)"
          f"{rate_on_line}")
    print(f"{'family':>8s} {'calibrated':<34s} {'g(0.05)':>9s} {'price of min(X, a)':>19s} {'unlimited':>11s}")
    for family in ["ccoc", "ph", "wang", "dual", "tvar"]:
        d = calibrate(family, x, premium, assets=assets)
        parameter = AGG[(example, f"param_{family}")]
        for sv in (0.05, 0.5):
            agg_g = AGG_G[family](parameter, sv)
            # The calibrated CCoC is the cost of capital the premium was built from: a consistency check.
            _record(s, f"{family}: g({sv}) at aggregate's parameter", agg_g, d.g(sv),
                    abs(d.g(sv) - agg_g) / agg_g, "mean", False, family == "ccoc")
        # The calibrated parameter, read off the distortion, against the one aggregate calibrated.
        [(key, value)] = d.parameters.items()  # each of these families has one parameter
        compare(s, f"{family}: parameter {key}", example, f"param_{family}", value, check=family == "ccoc")
        # Calibrated to the premium, so its price of min(X, a) is the premium: a consistency check.
        compare(s, f"{family}: price of min(X, a)", example, f"price_ask_{family}", d.measure(capped), check=True)
        if family != "ccoc":
            compare(s, f"{family}: unlimited price", example, f"unlimited_price_ask_{family}", d.measure(x),
                    show=False)
        priced[(view, family)] = d
        unlimited = f"{d.measure(x):11.4f}" if family != "ccoc" else f"{'n/a':>11s}"
        label = f"{d.family}(" + ", ".join(f"{k}={v:.6f}" for k, v in d.parameters.items()) + ")"
        print(f"{family:>8s} {label:<34s} {d.g(0.05):9.5f} {d.measure(capped):19.6f} {unlimited}")
    for name, d in FIXED:
        compare(s, f"fixed {name}: price of min(X, a)", example, f"fixed_price_ask_{name}", d.measure(capped),
                show=False, check=name.startswith("ccoc"))
    report()

The CCoC premium agrees to rounding, and so do the calibrated distortions: their parameters, read off with `Distortion.parameters`, are reproduced, and so is `g(s)` at `aggregate`'s parameter. The parameter and `g` rows, the unlimited prices and the fixed-parameter prices are the independent comparisons in this section. The other rows hold by construction and are counted as consistency checks: each calibrated price of `min(X, a)` is the premium it was calibrated to, the capital is `a - P`, the return on capital is the 10% the premium was built from, and the calibrated and fixed CCoC reproduce that 10%.

`aggregate` stops its Newton iteration at a premium error (4.3e-6 for the gross view's TVaR), where `prospicio` bisects to rounding, so a few calibrated parameters differ at the 1e-9 level rather than to rounding. TVaR is the clearest case: its premium moves by over a thousand per unit of its parameter, so that error moves the parameter, and `g`, only at the 1e-9 level. The TVaR parameter differs by 8.0e-9 on the gross view and 1.4e-9 on the ceded view, and its `g` rows by 4.9e-9 and 1.0e-9. The ceded view's proportional hazard is at the same level, from the same stopping rule: its parameter differs by 1.3e-9 and its `g(0.05)` by 2.8e-9. All of these are inside the same 1e-6 as the other families. It shows in `aggregate`'s price of `min(X, a)` for these distortions, which is off the target premium in the ninth digit where `prospicio`'s is not.

## Summary of every comparison

The table below lists every quantity compared in this notebook: `aggregate`'s value from the reference file, `prospicio`'s value, the difference (relative, or in buckets for quantiles) and whether it is inside the tolerance stated for its type. First, the totals by section.

In [ ]:
from collections import OrderedDict

independent = [r for r in COMPARISONS if not r["check"]]
checks = [r for r in COMPARISONS if r["check"]]

by_section = OrderedDict()
for r in independent:
    by_section.setdefault(r["section"], []).append(r)
print("independent comparisons by section")
print(f"{'section':32s} {'compared':>9s} {'passed':>7s}  worst relative difference")
for section, rows in by_section.items():
    worst = max((r for r in rows if r["unit"] == "rel"), key=lambda r: r["diff"])
    print(f"{section:32s} {len(rows):9d} {sum(r['ok'] for r in rows):7d}  {worst['diff']:.1e} ({worst['name']})")

print("\nindependent comparisons by tolerance class")
print(f"{'class':16s} {'tolerance':>12s} {'rows':>5s} {'worst':>12s}  where")
for cls, (tol, unit) in TOLERANCES.items():
    rows = [r for r in independent if r["cls"] == cls]
    worst = max(rows, key=lambda r: r["diff"])
    fmt = (lambda v: f"{v:.1e}") if unit == "rel" else (lambda v: f"{v:g} bkt")
    print(f"{cls:16s} {fmt(tol):>12s} {len(rows):5d} {fmt(worst['diff']):>12s}  {worst['section']}: {worst['name']}")

failed = [r for r in COMPARISONS if not r["ok"]]
print(f"\n{sum(r['ok'] for r in COMPARISONS)} of {len(COMPARISONS)} comparisons are inside their tolerance: "
      f"{len(independent)} independent comparisons and {len(checks)} consistency checks that hold by construction"
      + ("" if not failed else f"; outside: {[r['name'] for r in failed]}"))

In [ ]:
print(f"{'quantity':52s} {'aggregate':>15s} {'prospicio':>15s} {'difference':>11s} {'tolerance':>10s}  pass")
last = None
for r in COMPARISONS:
    if r["section"] != last:
        print(f"-- {r['section']}")
        last = r["section"]
    diff = f"{r['diff']:.1e}" if r["unit"] == "rel" else f"{r['diff']:.2f} bkt"
    tol = f"{r['tol']:.0e}" if r["unit"] == "rel" else f"{r['tol']} bkt"
    name = r["name"] + (" *" if r["check"] else "")
    print(f"{name:52s} {r['aggregate']:15.9g} {r['prospicio']:15.9g} {diff:>11s} {tol:>10s}  "
          f"{'yes' if r['ok'] else 'NO'}")

## Where and why the numbers differ

Nothing above is outside its tolerance. The tolerances were set after a first run, from the differences explained here; the summary prints the worst observed difference in each class beside its tolerance, and most results agree far better. What differences there are come from six places.

1. **The grid.** The bucket size sets every grid number: the severity is rounded to it and the loss beyond a limit sits on the limit. `prospicio` takes the bucket from the caller, so each section passes the one `aggregate` chose, for parity only. Left to `recommend_grid` or `fft_auto` the answer can differ: for section 1's book they see only the unlimited lognormal and recommend a step of 2, where `aggregate`, which sees the policy limit, uses 4. A finer step is the safer choice, but its numbers then differ from `aggregate`'s by discretisation. On a shared grid the quantiles agree exactly: at every compared quantile the cdf is at least 6.7e-8 away from the probability, far above rounding, so none sits on a tie.
2. **The grid mean is not the exact mean.** `est_m` in `aggregate` and `Grid.mean` here are the mean of the rounded severity compounded: 950.16 against an exact 950.58 in section 1, 0.044% low, and 544.17 against 544.59 for the layer 250 xs 0. Contract terms are exact for the discretized severity, not for the continuous one. Quote the exact value (`Lognormal.lev`, `Mixture.lev`, `layer`) when it matters, and compare grid with grid, as the tables do.
3. **A share off the grid.** `aggregate` re-buckets a ceded amount between its buckets, which adds a little variance; `prospicio` gives a share its own step (`share x bs`), which adds no rounding of its own. Means agree; the standard deviation differs by 2.6e-6 for the 80% share and 7.4e-6 for the 50% stop-loss share, the skewness by up to 9.5e-6, and a quantile by at most half a bucket (sections 2 and 4).
4. **The tail beyond the grid.** `prospicio` puts the probability beyond the last grid point on that point; `aggregate` (with `normalize=False`) drops it. The effect is the probability beyond the grid times the distance to the layer, which is why the stop loss at 2,800 deep in section 3's tail carries a 1e-5 tolerance (5.7e-7 observed). Unnormalised is the setting that matches; `aggregate`'s default `normalize=True` thins the tail instead.
5. **Skewness.** An FFT leaves rounding noise of order 1e-16 on every point of the grid, up to its top at `2**16 x bs` (262,144 for a bucket of 4), and the two packages leave different noise. The third moment weights a point by the cube of its distance from the mean, so noise at the top of the grid moves the skewness of a light-tailed cover in the sixth digit. Against the skewness of `aggregate`'s density (`*_skew_central` in the reference file) the layer 250 xs 250 differs by about 2e-6 (1.5e-6 on Windows, 1.8e-6 on Linux: the noise depends on the platform); dropping `prospicio`'s probability beyond 10,000, which that layer's annual loss cannot reach with any real probability, brings it under 1e-13 (the cell below). The skew that `aggregate` quotes (`*_skew`) differs again, by up to 8.1e-5: `xsden_to_meancvskew` places the density's rounding deficit (`1 - sum(p)`, of order 1e-15 to 1e-14) one bucket past the grid. That is a convention, not a property of the distribution, so the notebook compares the density's skewness, with a 2e-5 tolerance. The worst difference observed, 9.5e-6, is the 50% stop-loss share of point 3, where `aggregate`'s re-bucketing adds to the noise. Skewness is the one statistic whose tolerance rests on a modelling choice rather than on the grid.
6. **Definitions.** A few quantities mean something specific in `aggregate`, and the comparison uses the same meaning: its quantiles are the lower quantile on the grid; `cond_sev_mean` and the attachment probability are conditional on the claim reaching the layer; `net of` holds the loss on the *other* side of the layer; the "insurance charge" of an aggregate deductible is the excess ratio (the Table M charge), not the loss elimination ratio; the probability of a reinstatement is the probability of any loss to the layer; reinstatements are pro rata as to amount only (`prospicio` also has them pro rata as to time, which is not compared); a swing premium's terms are scaled by the placed share, but not its loss conversion factor; a profit commission does not deduct a ceding commission; and the calibration of the distortions stops at a premium error.

Not compared: bid prices (they need the dual distortion applied through `aggregate`'s `Distortion.price`), the Gini and the standard deviation of the reinstatement premium. The exposure-rated cat and casualty examples, the loss corridor (the two packages define it differently when an annual limit exists) and multi-line limit profiles are left out; see `knowledge/references/aggregate-mildenhall.md`.

In [ ]:
quoted = [(k, AGG[k] / AGG[(k[0], k[1] + "_central")] - 1.0) for k in AGG
          if k[1].endswith("_skew") and (k[0], k[1] + "_central") in AGG]
quoted.sort(key=lambda kv: -abs(kv[1]))
print(f"{len(quoted)} skewness values; aggregate's quoted skew is within {abs(quoted[0][1]):.1e} of its density's.")
print("largest differences, with the density deficit that aggregate places one bucket past the grid:")
for (example, quantity), diff in quoted[:4]:
    deficit = AGG[(example, quantity.replace("_skew", "_density_deficit"))]
    beyond = POINTS * AGG[(example, "bs")]  # one bucket past the grid's top
    print(f"  {example} {quantity}: {diff:+.1e} (deficit {deficit:.1e} at {beyond:,.0f} adds about "
          f"{deficit * beyond**3:,.0f} to the third moment)")

# prospicio's skewness of 250 xs 250 against aggregate's density, with and without the noise at the top of the grid.
layer = tower_grids.ceded[1]
central = AGG[("tower_gross", "layer_250x250_ceded_skew_central")]
kept = [(i * layer.step, q) for i, q in enumerate(layer.probs) if i * layer.step < 10000.0]
mean = sum(x * q for x, q in kept)
var = sum((x - mean) ** 2 * q for x, q in kept)
below = sum((x - mean) ** 3 * q for x, q in kept) / var**1.5
whole = moments(layer)[3]
print(f"\nlayer 250 xs 250, skewness of aggregate's density {central:.10f}; prospicio's on the whole grid "
      f"{whole:.10f} ({whole / central - 1:+.1e}), below 10,000 {below:.10f} ({below / central - 1:+.1e}); "
      f"probability dropped {sum(q for i, q in enumerate(layer.probs) if i * layer.step >= 10000.0):.1e}")